# Задание 1. Разведывательный анализ (EDA) и предобработка
Проверьте данные на пропуски. Обоснуйте метод их заполнения (LOCF, интерполяция и т.д.), учитывая риск упреждения (look-ahead bias).
Постройте графики временного ряда, опишите визуально наблюдаемые особенности.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
SOURCE_FILEPATH = Path("data.parquet")
df = pd.read_csv(SOURCE_FILEPATH, parse_dates=["TS"])

df["TS"] = pd.to_datetime(df["TS"], utc=True, format="mixed")

df = df.sort_values("TS").reset_index(drop=True)

df.head()

,row,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,1,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,3,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,4,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,5,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


In [3]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()

# row - служебная колонка, ее не рассматриваем как временной ряд.
numeric_cols = [col for col in numeric_cols if col != "row"]

print(f"Найдено числовых временных рядов: {len(numeric_cols)}")

Найдено числовых временных рядов: 117


## Есть ли пропуски?

In [4]:
# Проверка пропусков и константных признаков

missing = df.isna().sum()
missing_nonzero = missing[missing > 0]

nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique == 1].index.tolist()

print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"Всего формальных пропусков NaN/None/NaT: {missing.sum()}")
print(f"Колонок с пропусками: {len(missing_nonzero)}")
print(f"Константных колонок: {len(constant_cols)}")

constant_report = pd.DataFrame({
    "column": constant_cols,
    "constant_value": [df[col].iloc[0] for col in constant_cols],
})

display(constant_report)

Размер данных: 2878 строк, 119 колонок
Всего формальных пропусков NaN/None/NaT: 0
Колонок с пропусками: 0
Константных колонок: 33


,column,constant_value
0,Lab1_G2_Fнш,0.00
1,Lab1_G2_Fма,0.00
2,Lab1_G2_Fмн,0.00
3,Lab1_G2_Fств,0.00
4,Lab1_G3_Маслосистема,0.00
5,Lab1_Lmp_Txt_AO,0.00
6,Lab1_Lmp_Txt_BEAO,0.00
7,Lab1_TC_dPvfKVOU,0.60
8,Lab1_Lmp_Txt_BK1,0.00
9,Lab1_Lmp_Txt_BK2,0.00


## Есть ли дубли?

In [5]:
# Проверка полных дублей строк
n_full_duplicates = df.drop(columns=["row"]).duplicated().sum()

print(f"Полных дублей строк: {n_full_duplicates}")

# Проверка дублей по временной метке

ts_duplicates_count = df.duplicated(subset=["TS"]).sum()

print(f"Дублей по TS: {ts_duplicates_count}")

df[df.duplicated(subset=["TS"], keep=False)].sort_values("TS").head()

if n_full_duplicates == 0:
    print("Полные дубли строк отсутствуют.")
else:
    print("Обнаружены полные дубли строк, требуется дополнительная проверка.")
    duplicates = df[df.drop(columns=["row"]).duplicated(keep=False)]
    print(duplicates)
    

Полных дублей строк: 1439
Дублей по TS: 1439
Обнаружены полные дубли строк, требуется дополнительная проверка.
       row                               TS  Lab1_G1_N1  Lab1_G1_N2  \
0        1 2022-12-02 18:59:59.999998+00:00        8726       11490   
1        2 2022-12-02 18:59:59.999998+00:00        8726       11490   
2        3 2022-12-02 19:00:59.999998+00:00        8726       11489   
3        4 2022-12-02 19:00:59.999998+00:00        8726       11489   
4        5 2022-12-02 19:01:59.999999+00:00        8725       11493   
...    ...                              ...         ...         ...   
2873  2874        2022-12-03 18:56:00+00:00        8686       11447   
2874  2875 2022-12-03 18:56:59.999999+00:00        8690       11449   
2875  2876 2022-12-03 18:56:59.999999+00:00        8690       11449   
2876  2877        2022-12-03 18:58:00+00:00        8690       11449   
2877  2878        2022-12-03 18:58:00+00:00        8690       11449   

      Lab1_G1_N3  Lab1_G1_P2  Lab1_G

## Удаляем полные дубли

In [6]:
df = (
    df
    .drop(columns=["row"])
    .drop_duplicates()
    .reset_index(drop=True)
)

# Факторизация временных рядов

Каждый числовой столбец датасета рассматривается как отдельный временной ряд.

Сначала выделяем:
- константные ряды;
- неконстантные ряды.

Константные ряды рассматриваются отдельно: два константных ряда считаются эквивалентными, если имеют одно и то же постоянное значение.

Задаём множество временных рядов
один столбец = один временной ряд

In [7]:
# Формируем множество временных рядов.
# Отношение эквивалентности для константных рядов:
# xi ~ xj, если xi(t) = xj(t) для всех t.
#
# То есть константные ряды попадают в один класс,
# если имеют одно и то же постоянное значение.


service_cols = {"TS", "row"}

series_cols = [
    col for col in df.select_dtypes(include="number").columns
    if col not in service_cols
]

# Делим ряды на константные и неконстантные.
nunique = df[series_cols].nunique(dropna=True)

constant_cols = nunique[nunique == 1].index.tolist()
nonconstant_cols = nunique[nunique > 1].index.tolist()

print("Всего временных рядов:", len(series_cols))
print("Константных рядов:", len(constant_cols))
print("Неконстантных рядов:", len(nonconstant_cols))


# Константные ряды группируем по постоянному значению.
constant_report = (
    pd.DataFrame({
        "series_name": constant_cols,
        "constant_value": [df[col].dropna().iloc[0] for col in constant_cols],
    })
    .groupby("constant_value", as_index=False)
    .agg(
        members=("series_name", list),
        class_size=("series_name", "count"),
    )
)

constant_report["representative"] = constant_report["members"].apply(lambda x: x[0])
constant_report["group"] = "constant"

constant_representatives = constant_report["representative"].tolist()

display(constant_report)

print("Классов константных рядов:", len(constant_report))

Всего временных рядов: 117
Константных рядов: 33
Неконстантных рядов: 84


,constant_value,members,class_size,representative,group
0,0.00,"[Lab1_G2_Fнш, Lab1_G2_Fма, Lab1_G2_Fмн, Lab1_G...",17,Lab1_G2_Fнш,constant
1,0.27,[Lab1_hGPA],1,Lab1_hGPA,constant
2,0.60,[Lab1_TC_dPvfKVOU],1,Lab1_TC_dPvfKVOU,constant
3,0.92,[Lab1_Kp],1,Lab1_Kp,constant
4,1.00,"[Lab1_TC_VKPGV, Lab1_TC_VPOS, Lab1_Kran_10, La...",11,Lab1_TC_VKPGV,constant
5,26.00,[Lab1_Tpg],1,Lab1_Tpg,constant
6,50.80,[Lab1_q],1,Lab1_q,constant


Классов константных рядов: 7


In [8]:
print("Отношение эквивалентности для константных рядов:")
print("xi ~ xj, если xi(t) = xj(t) для всех t")
print()

print("Классы эквивалентности для константных рядов:")

for value, members in constant_classes.items():
    print(f"Значение {value}: {len(members)} рядов")
    print(members)
    print()

Отношение эквивалентности для константных рядов:
xi ~ xj, если xi(t) = xj(t) для всех t

Классы эквивалентности для константных рядов:


NameError: name 'constant_classes' is not defined

## Проверка стационарности

Стационарный временной ряд - это ряд, у которого со временем не меняются базовые свойства: средний уровень, разброс и характер колебаний. На практике это значит, что ряд не должен постоянно уплывать вверх или вниз и не должен менять масштаб колебаний от одного участка к другому.

Для проверки используем два стандартных теста: ADF и KPSS. Они хорошо дополняют друг друга, потому что у них противоположные нулевые гипотезы. Описание этих тестов и примеры есть в документации `statsmodels`: https://www.statsmodels.org/stable/examples/notebooks/generated/stationarity_detrending_adf_kpss.html

### ADF (Augmented Dickey Fuller)

ADF-тест проверяет, есть ли у ряда **единичный корень**.

?: если у ряда есть единичный корень, то случайные изменения в нём не затухают. Сдвиг или скачок остаётся в памяти ряда, и ряд может долго блуждать. Такой ряд обычно нестационарен: его среднее значение плохо фиксируется около одного уровня.

Для ADF:

- нулевая гипотеза: у ряда есть единичный корень, ряд нестационарен;
- если `p-value < alpha`, нулевая гипотеза отвергается;
- тогда по ADF ряд можно считать стационарным.

### KPSS (Kwiatkowski-Phillips-Schmidt-Shin)

KPSS устроен наоборот.

Для KPSS:

- нулевая гипотеза: ряд стационарен;
- если `p-value >= alpha`, нет оснований отвергать стационарность;
- тогда по KPSS ряд можно считать стационарным.

### Уровень значимости

В расчётах берём `alpha = 0.05` (выбрали эмпирически).

Итоговая классификация:

| ADF | KPSS | Интерпретация |
|---|---|---|
| стационарен | стационарен | `stationary` |
| стационарен | нестационарен | `difference_stationary_or_borderline` |
| нестационарен | стационарен | `trend_stationary_or_borderline` |
| нестационарен | нестационарен | `nonstationary` |

Дальше стационарные ряды факторизируем отдельно. Остальные ряды разбираем по типу нестационарности.

In [9]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning

import warnings


ALPHA = 0.05


def check_stationarity(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    # ADF: H0 - ряд нестационарен.
    adf_result = adfuller(values, autolag="AIC")
    adf_pvalue = adf_result[1]
    adf_stationary = adf_pvalue < ALPHA

    # KPSS: H0 - ряд стационарен.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)
        kpss_result = kpss(values, regression="c", nlags="auto")

    kpss_pvalue = kpss_result[1]
    kpss_stationary = kpss_pvalue >= ALPHA

    if adf_stationary and kpss_stationary:
        stationarity_type = "stationary"
    elif adf_stationary and not kpss_stationary:
        stationarity_type = "difference_stationary_or_borderline"
    elif not adf_stationary and kpss_stationary:
        stationarity_type = "trend_stationary_or_borderline"
    else:
        stationarity_type = "nonstationary"

    return {
        "adf_pvalue": adf_pvalue,
        "adf_stationary": adf_stationary,
        "kpss_pvalue": kpss_pvalue,
        "kpss_stationary": kpss_stationary,
        "stationarity_type": stationarity_type,
    }


stationarity_rows = []

for col in nonconstant_cols:
    result = check_stationarity(df[col])
    result["series_name"] = col
    stationarity_rows.append(result)

stationarity_report = pd.DataFrame(stationarity_rows)

stationary_cols = stationarity_report[
    stationarity_report["stationarity_type"] == "stationary"
]["series_name"].tolist()

difference_cols = stationarity_report[
    stationarity_report["stationarity_type"] == "difference_stationary_or_borderline"
]["series_name"].tolist()

trend_borderline_cols = stationarity_report[
    stationarity_report["stationarity_type"] == "trend_stationary_or_borderline"
]["series_name"].tolist()

nonstationary_cols = stationarity_report[
    stationarity_report["stationarity_type"] == "nonstationary"
]["series_name"].tolist()

display(stationarity_report)

display(
    stationarity_report["stationarity_type"]
    .value_counts()
    .to_frame("count")
)

print("Стационарных рядов:", len(stationary_cols))
print("Difference-stationary / borderline:", len(difference_cols))
print("Trend-stationary / borderline:", len(trend_borderline_cols))
print("Нестационарных рядов:", len(nonstationary_cols))

,adf_pvalue,adf_stationary,kpss_pvalue,kpss_stationary,stationarity_type,series_name
0,5.097505e-03,True,0.010000,False,difference_stationary_or_borderline,Lab1_G1_N1
1,3.097609e-02,True,0.011368,False,difference_stationary_or_borderline,Lab1_G1_N2
2,1.416737e-25,True,0.100000,True,stationary,Lab1_G1_N3
3,1.412013e-02,True,0.010000,False,difference_stationary_or_borderline,Lab1_G1_P2
4,4.209119e-02,True,0.021574,False,difference_stationary_or_borderline,Lab1_G1_T4ср
...,...,...,...,...,...,...
79,9.903337e-01,False,0.010000,False,nonstationary,Lab1_TposleNag
80,8.585527e-01,False,0.010000,False,nonstationary,Lab1_PdoNag
81,9.446488e-01,False,0.010000,False,nonstationary,Lab1_TdoNag
82,9.621477e-01,False,0.010000,False,nonstationary,Lab1_Rc


,count
stationarity_type,
difference_stationary_or_borderline,36
stationary,23
nonstationary,23
trend_stationary_or_borderline,2


Стационарных рядов: 23
Difference-stationary / borderline: 36
Trend-stationary / borderline: 2
Нестационарных рядов: 23


In [10]:
stationary_only = stationarity_report[
    stationarity_report["stationarity_type"] == "stationary"
].copy()

print("Количество стационарных рядов:", len(stationary_only))

display(stationary_only)

Количество стационарных рядов: 23


,adf_pvalue,adf_stationary,kpss_pvalue,kpss_stationary,stationarity_type,series_name
2,1.416737e-25,True,0.100000,True,stationary,Lab1_G1_N3
12,6.523998e-05,True,0.100000,True,stationary,Lab1_G2_Fc2
14,1.255445e-06,True,0.100000,True,stationary,Lab1_G2_Fc3
16,1.246249e-02,True,0.061385,True,stationary,Lab1_G2_3F1
17,0.000000e+00,True,0.085633,True,stationary,Lab1_G2_Fтк2
20,7.032420e-09,True,0.100000,True,stationary,Lab1_G2_F2
30,1.470105e-04,True,0.100000,True,stationary,Lab1_G2_2F3
33,0.000000e+00,True,0.100000,True,stationary,Lab1_G2_Fтк8
34,8.132212e-28,True,0.100000,True,stationary,Lab1_G2_Fн9
35,5.546521e-03,True,0.100000,True,stationary,Lab1_G2_Fв9


In [11]:
cols_for_factorization = stationary_cols

In [12]:
# Факторизируем только стационарные ряды,
# которые прошли оба теста: ADF и KPSS.

cols_for_factorization = stationary_cols.copy()

print("Количество стационарных рядов для факторизации:", len(cols_for_factorization))
print(cols_for_factorization)

Количество стационарных рядов для факторизации: 23
['Lab1_G1_N3', 'Lab1_G2_Fc2', 'Lab1_G2_Fc3', 'Lab1_G2_3F1', 'Lab1_G2_Fтк2', 'Lab1_G2_F2', 'Lab1_G2_2F3', 'Lab1_G2_Fтк8', 'Lab1_G2_Fн9', 'Lab1_G2_Fв9', 'Lab1_G3_N3', 'Lab1_G3_Lm', 'Lab1_G3_Pm', 'Lab1_G3_T638', 'Lab1_G3_V1', 'Lab1_G3_V2', 'Lab1_G3_Pc1', 'Lab1_G3_КНД', 'Lab1_G3_КВД', 'Lab1_G3_Турбина_ГГ', 'Lab1_G3_ПО_СТ', 'Lab1_G3_ЗО_СТ', 'Lab1_TC_P615']


### Проверка факторизации по форме ряда

Сначала для стационарных рядов была проверена более строгая идея факторизации: сравнивать сами нормированные временные ряды.

Каждый ряд нормировался в диапазон от 0 до 1, после чего для пары рядов вычислялось среднее абсолютное расстояние между значениями в одинаковые моменты времени.

Такой подход позволяет найти почти совпадающие по форме ряды. Однако он может быть слишком строгим: два стационарных процесса могут иметь похожие статистические свойства, но не совпадать точка в точку.

In [13]:
###### Блок вспомогательных функций для факторизаци##########
def min_max_scale(df_features, feature_cols):
    # Нормировка выбранных признаков в диапазон [0, 1].
    scaled = df_features.copy()

    for col in feature_cols:
        min_value = scaled[col].min()
        max_value = scaled[col].max()
        amplitude = max_value - min_value

        if amplitude == 0:
            scaled[col] = 0
        else:
            scaled[col] = (scaled[col] - min_value) / amplitude

    return scaled


def mean_abs_distance(features_df, feature_cols, series_1, series_2):
    # Среднее абсолютное расстояние между двумя объектами
    # по выбранным нормированным признакам.
    row_1 = features_df.loc[
        features_df["series_name"] == series_1,
        feature_cols
    ].iloc[0]

    row_2 = features_df.loc[
        features_df["series_name"] == series_2,
        feature_cols
    ].iloc[0]

    return np.mean(np.abs(row_1 - row_2))


def build_classes(cols, distance_func, eps):
    # Построение графа похожести.
    # Вершины — временные ряды, ребро — расстояние не превышает eps.
    edges = []

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            col1 = cols[i]
            col2 = cols[j]

            distance = distance_func(col1, col2)

            if distance <= eps:
                edges.append({
                    "series_1": col1,
                    "series_2": col2,
                    "distance": distance,
                })

    edges_df = pd.DataFrame(edges)

    graph = {col: [] for col in cols}

    for _, row in edges_df.iterrows():
        graph[row["series_1"]].append(row["series_2"])
        graph[row["series_2"]].append(row["series_1"])

    # Связные компоненты графа задают классы эквивалентности.
    visited = set()
    classes = []

    for col in cols:
        if col in visited:
            continue

        stack = [col]
        current_class = []

        while stack:
            current = stack.pop()

            if current in visited:
                continue

            visited.add(current)
            current_class.append(current)

            for neighbor in graph[current]:
                if neighbor not in visited:
                    stack.append(neighbor)

        classes.append(current_class)

    return classes, edges_df


def choose_representatives(classes, distance_func=None):
    # Выбор представителя для каждого класса.
    # Для нетривиального класса выбирается объект с минимальным
    # средним расстоянием до остальных элементов класса.
    representatives = []
    rows = []

    for class_id, members in enumerate(classes, start=1):
        if len(members) == 1 or distance_func is None:
            representative = members[0]
            mean_distance = 0.0
        else:
            scores = {}

            for col in members:
                distances = [
                    distance_func(col, other_col)
                    for other_col in members
                    if col != other_col
                ]

                scores[col] = np.mean(distances)

            representative = min(scores, key=scores.get)
            mean_distance = scores[representative]

        representatives.append(representative)

        rows.append({
            "local_class_id": class_id,
            "representative": representative,
            "class_size": len(members),
            "mean_distance_representative": mean_distance,
            "members": members,
        })

    return representatives, pd.DataFrame(rows)

In [14]:
def normalize_to_01(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    amplitude = values.max() - values.min()

    if amplitude == 0:
        return np.zeros_like(values)

    return (values - values.min()) / amplitude


stationary_normalized = {
    col: normalize_to_01(df[col])
    for col in stationary_cols
}


def shape_distance(series_1, series_2):
    x = stationary_normalized[series_1]
    y = stationary_normalized[series_2]

    n = min(len(x), len(y))

    return np.mean(np.abs(x[:n] - y[:n]))


EPS_SHAPE = 0.05

shape_classes, shape_edges = build_classes(
    stationary_cols,
    shape_distance,
    EPS_SHAPE,
)

shape_class_sizes = [len(c) for c in shape_classes]

shape_report = pd.DataFrame([{
    "eps": EPS_SHAPE,
    "edges_count": len(shape_edges),
    "classes_count": len(shape_classes),
    "max_class_size": max(shape_class_sizes),
    "nontrivial_classes": sum(size > 1 for size in shape_class_sizes),
    "reduction": len(stationary_cols) - len(shape_classes),
}])

display(shape_report)

,eps,edges_count,classes_count,max_class_size,nontrivial_classes,reduction
0,0.05,11,18,5,2,5


По результатам проверки сравнение формы оказалось слишком строгим: оно объединяет только почти синхронно совпадающие ряды. Поэтому для основной факторизации стационарных рядов далее используется сравнение не исходных значений, а статистических характеристик ряда.

## Факторизация стационарных рядов

Стационарные ряды сравниваются по статистическим характеристикам. 

Для каждого стационарного ряда строится вектор характеристик:

- среднее значение;
- стандартное отклонение;
- амплитуда;
- нижний квартиль;
- медиана;
- верхний квартиль;
- автокорреляции с лагами 1, 5 и 10.

каждый временной ряд заменяется набором признаков:

`ряд -> [mean, std, amplitude, q25, median, q75, acf1, acf5, acf10]`

Так как признаки имеют разные масштабы, перед сравнением они нормируются в диапазон от 0 до 1. После нормировки между двумя рядами вычисляется расстояние как среднее абсолютное отличие между их векторами характеристик.

Далее строится граф похожести:

- вершины графа — стационарные временные ряды;
- ребро между двумя вершинами проводится, если расстояние между соответствующими рядами не превышает `eps = 0.05`.

После построения графа классы эквивалентности определяются как его связные компоненты. Это означает, что ряды попадают в один класс, если между ними есть путь по рёбрам похожести.

Таким образом, факторизация стационарных рядов состоит из следующих шагов:

1. построить вектор характеристик для каждого ряда;
2. нормировать признаки;
3. посчитать расстояния между рядами;
4. построить граф похожести;
5. взять связные компоненты графа как классы эквивалентности;
6. выбрать представителя каждого класса.

In [15]:
def autocorr(values, lag):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    if len(values) <= lag:
        return np.nan

    x = values[:-lag]
    y = values[lag:]

    if x.std() == 0 or y.std() == 0:
        return np.nan

    return np.corrcoef(x, y)[0, 1]


stationary_feature_rows = []

for col in stationary_cols:
    values = df[col].dropna().to_numpy(dtype=float)

    stationary_feature_rows.append({
        "series_name": col,
        "mean": values.mean(),
        "std": values.std(),
        "amplitude": values.max() - values.min(),
        "q25": np.quantile(values, 0.25),
        "median": np.quantile(values, 0.50),
        "q75": np.quantile(values, 0.75),
        "acf1": autocorr(values, 1),
        "acf5": autocorr(values, 5),
        "acf10": autocorr(values, 10),
    })

stationary_features = pd.DataFrame(stationary_feature_rows)

stationary_feature_cols = [
    "mean",
    "std",
    "amplitude",
    "q25",
    "median",
    "q75",
    "acf1",
    "acf5",
    "acf10",
]

stationary_features_norm = min_max_scale(
    stationary_features,
    stationary_feature_cols
)

display(stationary_features)

,series_name,mean,std,amplitude,q25,median,q75,acf1,acf5,acf10
0,Lab1_G1_N3,5000.548992,4.681688,27.000000,4998.000000,5001.000000,5004.000000,0.148135,-0.035206,-0.021598
1,Lab1_G2_Fc2,0.402967,0.103342,0.790000,0.330000,0.390000,0.460000,0.228567,0.134961,0.102971
2,Lab1_G2_Fc3,0.152960,0.049196,0.340000,0.120000,0.140000,0.170000,0.551173,0.470885,0.377544
3,Lab1_G2_3F1,0.207797,0.064000,0.340000,0.160000,0.210000,0.250000,0.831608,0.705996,0.622813
4,Lab1_G2_Fтк2,0.065413,0.014789,0.090000,0.050000,0.060000,0.070000,0.116033,-0.017700,0.015157
5,Lab1_G2_F2,2.914517,0.145252,0.900000,2.820000,2.920000,3.010000,0.858716,0.664033,0.518158
6,Lab1_G2_2F3,0.221230,0.037003,0.230000,0.200000,0.220000,0.250000,0.508248,0.397362,0.371281
7,Lab1_G2_Fтк8,0.068158,0.015927,0.120000,0.060000,0.070000,0.080000,0.065747,0.059084,0.004925
8,Lab1_G2_Fн9,0.108763,0.025764,0.200000,0.090000,0.110000,0.120000,0.222324,0.115271,0.093340
9,Lab1_G2_Fв9,0.031835,0.013363,0.100000,0.020000,0.030000,0.040000,0.524563,0.448928,0.432878


In [16]:
def stationary_distance(series_1, series_2):
    return mean_abs_distance(
        stationary_features_norm,
        stationary_feature_cols,
        series_1,
        series_2,
    )


EPS_STATIONARY = 0.05

stationary_classes, stationary_edges = build_classes(
    stationary_cols,
    stationary_distance,
    EPS_STATIONARY,
)

stationary_representatives, stationary_report = choose_representatives(
    stationary_classes,
    stationary_distance,
)

stationary_report["group"] = "stationary"

display(stationary_edges)

print("Факторизация стационарных рядов")
print("eps:", EPS_STATIONARY)
print("Исходных рядов:", len(stationary_cols))
print("Классов:", len(stationary_classes))
print("Сокращение:", len(stationary_cols) - len(stationary_classes))
print()

for i, members in enumerate(stationary_classes, start=1):
    print(f"Класс {i}: {len(members)} рядов")
    print(members)
    print()

,series_1,series_2,distance
0,Lab1_G2_Fc2,Lab1_G2_Fтк8,0.048510
1,Lab1_G2_Fc2,Lab1_G2_Fн9,0.008845
2,Lab1_G2_Fc2,Lab1_G3_V2,0.043666
3,Lab1_G2_Fc3,Lab1_G2_2F3,0.015686
4,Lab1_G2_Fc3,Lab1_G2_Fв9,0.015970
5,Lab1_G2_3F1,Lab1_G2_F2,0.028379
6,Lab1_G2_3F1,Lab1_G3_T638,0.049022
7,Lab1_G2_3F1,Lab1_G3_Турбина_ГГ,0.028686
8,Lab1_G2_Fтк2,Lab1_G2_Fтк8,0.016970
9,Lab1_G2_Fтк2,Lab1_G2_Fн9,0.041209


Факторизация стационарных рядов
eps: 0.05
Исходных рядов: 23
Классов: 4
Сокращение: 19

Класс 1: 1 рядов
['Lab1_G1_N3']

Класс 2: 12 рядов
['Lab1_G2_Fc2', 'Lab1_G3_V2', 'Lab1_G3_КВД', 'Lab1_G3_КНД', 'Lab1_G3_Lm', 'Lab1_G2_Fн9', 'Lab1_G3_Pm', 'Lab1_TC_P615', 'Lab1_G3_Pc1', 'Lab1_G3_N3', 'Lab1_G2_Fтк8', 'Lab1_G2_Fтк2']

Класс 3: 3 рядов
['Lab1_G2_Fc3', 'Lab1_G2_Fв9', 'Lab1_G2_2F3']

Класс 4: 7 рядов
['Lab1_G2_3F1', 'Lab1_G3_Турбина_ГГ', 'Lab1_G3_ПО_СТ', 'Lab1_G3_ЗО_СТ', 'Lab1_G3_V1', 'Lab1_G2_F2', 'Lab1_G3_T638']



## Факторизация рядов по приращениям

Для группы `difference_stationary_or_borderline` исходные значения ряда могут быть нестабильными. Поэтому вместо исходного временного ряда рассматривается новый ряд — ряд приращений.

Если исходный ряд имеет вид:

`x = [x1, x2, x3, ..., xn]`

то ряд приращений строится так:

`dx = [x2 - x1, x3 - x2, x4 - x3, ..., xn - x(n-1)]`

То есть элементами нового ряда являются разности соседних элементов исходного ряда. Такой ряд показывает не абсолютный уровень процесса, а то, как процесс изменяется от одного момента времени к следующему.

Далее статистические характеристики считаются именно для ряда приращений `dx`, а не для исходного ряда `x`.

Для каждого ряда приращений считаются:

- среднее приращение;
- стандартное отклонение приращений;
- амплитуда приращений;
- нижний квартиль приращений;
- медиана приращений;
- верхний квартиль приращений;
- среднее абсолютное приращение;
- доля положительных приращений.

После этого каждый исходный временной ряд представляется через характеристики своего ряда приращений:

`x -> dx -> [dx_mean, dx_std, dx_amplitude, dx_q25, dx_median, dx_q75, dx_abs_mean, dx_positive_share]`

Далее признаки нормируются в диапазон от 0 до 1, между рядами считается среднее абсолютное расстояние по этим признакам.

Граф похожести строится так же: вершины — исходные временные ряды, ребро проводится, если расстояние между характеристиками их рядов приращений не превышает `eps = 0.05`.

Связные компоненты графа считаются классами эквивалентности.

In [17]:
difference_cols = stationarity_report[
    stationarity_report["stationarity_type"] == "difference_stationary_or_borderline"
]["series_name"].tolist()

diff_feature_rows = []

for col in difference_cols:
    values = df[col].dropna().to_numpy(dtype=float)
    dx = np.diff(values)

    diff_feature_rows.append({
        "series_name": col,
        "dx_mean": dx.mean(),
        "dx_std": dx.std(),
        "dx_amplitude": dx.max() - dx.min(),
        "dx_q25": np.quantile(dx, 0.25),
        "dx_median": np.quantile(dx, 0.50),
        "dx_q75": np.quantile(dx, 0.75),
        "dx_abs_mean": np.mean(np.abs(dx)),
        "dx_positive_share": np.mean(dx > 0),
    })

diff_features = pd.DataFrame(diff_feature_rows)

diff_feature_cols = [
    "dx_mean",
    "dx_std",
    "dx_amplitude",
    "dx_q25",
    "dx_median",
    "dx_q75",
    "dx_abs_mean",
    "dx_positive_share",
]

diff_features_norm = min_max_scale(
    diff_features,
    diff_feature_cols
)

display(diff_features)

,series_name,dx_mean,dx_std,dx_amplitude,dx_q25,dx_median,dx_q75,dx_abs_mean,dx_positive_share
0,Lab1_G1_N1,-2.503477e-02,4.125896,48.000000,-2.000000,0.000000,2.000000,2.970793,0.429764
1,Lab1_G1_N2,-2.851182e-02,4.373538,95.000000,-2.000000,0.000000,2.000000,2.903338,0.417942
2,Lab1_G1_P2,-7.649513e-05,0.102814,2.440000,-0.010000,0.000000,0.010000,0.038630,0.380389
3,Lab1_G1_T4ср,-3.129346e-03,0.750844,18.900000,-0.400000,0.000000,0.400000,0.487135,0.437413
4,Lab1_G1_T606,-1.390821e-04,0.156855,1.500000,-0.100000,0.000000,0.100000,0.110292,0.320584
5,Lab1_G1_T1002,-2.086231e-04,0.077379,0.800000,0.000000,0.000000,0.000000,0.043324,0.178720
6,Lab1_G1_T1003,-2.781641e-04,0.230528,1.600000,-0.100000,0.000000,0.100000,0.167177,0.353964
7,Lab1_G2_F1,4.937413e-04,0.119525,0.990000,-0.070000,0.000000,0.070000,0.091203,0.449930
8,Lab1_G2_2F1,1.043115e-04,0.064414,0.560000,-0.040000,0.000000,0.040000,0.048588,0.436718
9,Lab1_G2_Fc4,6.954103e-06,0.120048,0.840000,-0.080000,0.000000,0.070000,0.091843,0.442281


In [18]:
def diff_distance(series_1, series_2):
    return mean_abs_distance(
        diff_features_norm,
        diff_feature_cols,
        series_1,
        series_2,
    )


EPS_DIFF = 0.05

diff_classes, diff_edges = build_classes(
    difference_cols,
    diff_distance,
    EPS_DIFF,
)

diff_representatives, diff_report = choose_representatives(
    diff_classes,
    diff_distance,
)

diff_report["group"] = "difference_stationary_or_borderline"

display(diff_edges)

print("Факторизация рядов по приращениям")
print("eps:", EPS_DIFF)
print("Исходных рядов:", len(difference_cols))
print("Классов:", len(diff_classes))
print("Сокращение:", len(difference_cols) - len(diff_classes))
print()

for i, members in enumerate(diff_classes, start=1):
    print(f"Класс {i}: {len(members)} рядов")
    print(members)
    print()

,series_1,series_2,distance
0,Lab1_G1_N2,Lab1_G4_N2,0.000000
1,Lab1_G1_P2,Lab1_G1_T606,0.033297
2,Lab1_G1_P2,Lab1_G1_T1003,0.026353
3,Lab1_G1_P2,Lab1_G2_F1,0.035296
4,Lab1_G1_P2,Lab1_G2_2F1,0.026173
...,...,...,...
280,Lab1_PologenieTRK,Lab1_TC_Pm,0.034360
281,Lab1_PologenieTRK,Lab1_Ne,0.024504
282,Lab1_TC_Pm,Lab1_Pm_sm_N,0.039461
283,Lab1_TC_Pm,Lab1_he,0.018436


Факторизация рядов по приращениям
eps: 0.05
Исходных рядов: 36
Классов: 6
Сокращение: 30

Класс 1: 1 рядов
['Lab1_G1_N1']

Класс 2: 2 рядов
['Lab1_G1_N2', 'Lab1_G4_N2']

Класс 3: 29 рядов
['Lab1_G1_P2', 'Lab1_Ne', 'Lab1_PologenieTRK', 'Lab1_TC_Pm', 'Lab1_he', 'Lab1_Pm_sm_N', 'Lab1_G4_Р2пр', 'Lab1_G4_delta_P2PR', 'Lab1_G4_T4пр', 'Lab1_G3_T1003', 'Lab1_G3_Pc3', 'Lab1_G3_dPf1', 'Lab1_G2_VoСТ', 'Lab1_G2_3F3', 'Lab1_G2_Fтк9', 'Lab1_G2_F3', 'Lab1_G2_Fс8', 'Lab1_G2_Fc9', 'Lab1_G2_3_77F2', 'Lab1_G2_Fтк4', 'Lab1_G2_3F2', 'Lab1_G2_Fкпа', 'Lab1_G2_Fцс', 'Lab1_G2_Fc4', 'Lab1_G2_2F1', 'Lab1_G2_F1', 'Lab1_G1_T1003', 'Lab1_G1_T606', 'Lab1_G1_T1002']

Класс 4: 2 рядов
['Lab1_G1_T4ср', 'Lab1_G4_T4']

Класс 5: 1 рядов
['Lab1_G3_T606']

Класс 6: 1 рядов
['Lab1_Qtg']



1. difference_stationary_or_borderline
   ADF считает стационарным, KPSS не считает.
   Такие ряды могут стать стационарными после взятия разностей.

2. trend_stationary_or_borderline
   KPSS считает стационарным, ADF не считает.
   Такие ряды могут иметь тренд, но после удаления тренда вести себя устойчиво.

3. nonstationary
   Оба теста не считают ряд стационарным.
   Их надо анализировать отдельно: скачки, тренды, приращения.

## Проверка нестационарных рядов

После обработки стационарных рядов и рядов по приращениям рассматривается оставшаяся группа рядов.

К нестационарным рядам добавляются также 2 ряда типа `trend_stationary_or_borderline`. Эти ряды не были отнесены к стационарным, так как результаты ADF и KPSS для них не совпали полностью. Поскольку таких рядов всего два, они не выделяются в отдельную группу, а рассматриваются вместе с нестационарными рядами.

Итоговая нестационарная группа состоит из:

- рядов типа `nonstationary`;
- рядов типа `trend_stationary_or_borderline`.

Нестационарные ряды могут отличаться по характеру динамики, поэтому перед сравнением делим их на три типа:

1. `trend` — ряды с выраженным направленным изменением уровня во времени.  
   Для таких рядов важны направление и сила тренда, а также изменение уровня от начала к концу ряда.

2. `jump` — ряды со скачкообразным изменением.  
   Для таких рядов важны момент скачка, величина скачка, направление скачка и то, насколько скачок выделяется на фоне обычных изменений.

3. `dynamic` — остальные нестационарные ряды без явно доминирующего тренда или скачка.  
   Для таких рядов требуется отдельный анализ динамики, поэтому на данном этапе они не объединяются.

После разделения на типы факторизация проверяется отдельно для групп `trend` и `jump`.

Для `trend`-рядов сравниваются признаки тренда:

- общее изменение по линейному тренду;
- абсолютная сила тренда;
- изменение уровня от начала к концу;
- начальный и конечный уровни.

Для `jump`-рядов сравниваются признаки скачка:

- относительное положение максимального скачка;
- величина и направление скачка;
- абсолютная величина скачка;
- доминирование скачка над обычными изменениями;
- начальный и конечный уровни.

Если при `eps = 0.05` связи похожести не возникают или почти не возникают, такие ряды не объединяются и остаются отдельными классами.

In [19]:
# 1. ряды типа nonstationary;
# 2. ряды типа trend_stationary_or_borderline.

nonstationary_group_cols = (
    nonstationary_cols
    + trend_borderline_cols
)

# Убираем возможные дубли, сохраняя порядок.
nonstationary_group_cols = list(dict.fromkeys(nonstationary_group_cols))

print("Нестационарных рядов:", len(nonstationary_cols))
print("Trend-stationary / borderline добавлено:", len(trend_borderline_cols))
print("Итого в общей группе:", len(nonstationary_group_cols))

Нестационарных рядов: 23
Trend-stationary / borderline добавлено: 2
Итого в общей группе: 25


Нестационарные ряды делятся на три типа по признакам нормированной динамики.

Сначала выделяются ряды со скачком (`jump`). Ряд относится к этому типу, если максимальный скачок составляет не менее 30% нормированной шкалы и при этом минимум в 8 раз больше обычного изменения ряда.

Если выраженного скачка нет, проверяется наличие тренда (`trend`). Ряд относится к трендовым, если линейный тренд или изменение уровня от начала к концу ряда достаточно велики.

Оставшиеся ряды относятся к типу `dynamic`. Это ряды без одного доминирующего скачка и без явно выраженного линейного тренда.

In [20]:



def get_nonstationary_features(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    eps = 1e-12
    n = len(values)

    amplitude = values.max() - values.min()

    if amplitude == 0:
        y = np.zeros_like(values)
    else:
        y = (values - values.min()) / amplitude

    dy = np.diff(y)

    # Линейный тренд на нормированном ряде.
    t = np.arange(n)
    slope = np.polyfit(t, y, 1)[0]
    trend_total_change = slope * n

    # Изменение уровня от начала к концу.
    level_change = y[-1] - y[0]

    # Максимальный скачок между соседними значениями.
    abs_dy = np.abs(dy)

    if len(abs_dy) == 0:
        jump_pos_rel = 0.0
        jump_size = 0.0
        jump_abs = 0.0
        jump_dominance = 0.0
    else:
        jump_pos = int(np.argmax(abs_dy) + 1)

        jump_pos_rel = jump_pos / n
        jump_size = dy[jump_pos - 1]
        jump_abs = abs(jump_size)
        jump_dominance = jump_abs / (np.median(abs_dy) + eps)

    return {
        "trend_total_change": trend_total_change,
        "trend_abs": abs(trend_total_change),
        "level_change": level_change,
        "level_abs_change": abs(level_change),
        "start_level": y[0],
        "end_level": y[-1],
        "jump_pos_rel": jump_pos_rel,
        "jump_size": jump_size,
        "jump_abs": jump_abs,
        "jump_dominance": jump_dominance,
    }


def classify_nonstationary_type(row):
    if row["jump_abs"] >= 0.30 and row["jump_dominance"] >= 8:
        return "jump"

    if row["trend_abs"] >= 0.30 or row["level_abs_change"] >= 0.40:
        return "trend"

    return "dynamic"


nonstationary_feature_rows = []

for col in nonstationary_group_cols:
    row = get_nonstationary_features(df[col])
    row["series_name"] = col
    nonstationary_feature_rows.append(row)

nonstationary_features = pd.DataFrame(nonstationary_feature_rows)

nonstationary_features["nonstationary_type"] = nonstationary_features.apply(
    classify_nonstationary_type,
    axis=1,
)

trend_cols = nonstationary_features[
    nonstationary_features["nonstationary_type"] == "trend"
]["series_name"].tolist()

jump_cols = nonstationary_features[
    nonstationary_features["nonstationary_type"] == "jump"
]["series_name"].tolist()

dynamic_cols = nonstationary_features[
    nonstationary_features["nonstationary_type"] == "dynamic"
]["series_name"].tolist()

display(
    nonstationary_features[
        [
            "series_name",
            "nonstationary_type",
            "trend_abs",
            "level_abs_change",
            "jump_abs",
            "jump_dominance",
            "jump_pos_rel",
        ]
    ]
)

display(
    nonstationary_features["nonstationary_type"]
    .value_counts()
    .to_frame("count")
)

print("trend:", len(trend_cols))
print("jump:", len(jump_cols))
print("dynamic:", len(dynamic_cols))

,series_name,nonstationary_type,trend_abs,level_abs_change,jump_abs,jump_dominance,jump_pos_rel
0,Lab1_G1_T1,dynamic,0.030671,0.087379,0.145631,1.500000e+01,0.028492
1,Lab1_G1_T607,trend,0.438969,0.125000,0.250000,2.500000e+11,0.526060
2,Lab1_G1_T600,dynamic,0.042507,0.100000,0.100000,1.000000e+11,0.865879
3,Lab1_G1_T638,dynamic,0.068331,0.133333,0.133333,1.333333e+11,0.985407
4,Lab1_G2_2F2,trend,0.446127,0.352941,0.500000,5.666667e+00,0.954135
5,Lab1_G3_Pc2,jump,0.068027,0.487563,0.346122,1.828941e+02,0.628214
6,Lab1_G3_T600,trend,0.182796,0.491605,0.137288,5.921179e+01,0.154274
7,Lab1_G3_T1002,trend,0.186038,0.558655,0.108283,5.019431e+01,0.454482
8,Lab1_G4_N1пр,trend,0.784421,0.460907,0.138939,8.672000e+00,0.360667
9,Lab1_G4_delta_T4PR,dynamic,0.212378,0.241056,0.204170,1.688776e+01,0.981932


,count
nonstationary_type,
trend,13
dynamic,6
jump,6


trend: 13
jump: 6
dynamic: 6


### Проверка трендовых рядов

Для трендовых рядов сравниваются признаки, связанные с направленным изменением уровня:

- общее изменение по линейному тренду;
- абсолютная сила тренда;
- изменение от начала к концу ряда;
- начальный и конечный уровни.

Если расстояние между двумя рядами по этим признакам не превышает `eps = 0.05`, между ними проводится ребро в графе похожести.

In [21]:
trend_cols = nonstationary_features[
    nonstationary_features["nonstationary_type"] == "trend"
]["series_name"].tolist()

trend_feature_cols = [
    "trend_total_change",
    "trend_abs",
    "level_change",
    "level_abs_change",
    "start_level",
    "end_level",
]

trend_features = nonstationary_features[
    nonstationary_features["series_name"].isin(trend_cols)
][["series_name"] + trend_feature_cols].copy()

trend_features_norm = min_max_scale(
    trend_features,
    trend_feature_cols
)


def trend_distance(series_1, series_2):
    return mean_abs_distance(
        trend_features_norm,
        trend_feature_cols,
        series_1,
        series_2,
    )


EPS_TREND = 0.05

trend_classes, trend_edges = build_classes(
    trend_cols,
    trend_distance,
    EPS_TREND,
)

trend_representatives, trend_report = choose_representatives(
    trend_classes,
    trend_distance,
)

trend_report["group"] = "trend"

display(trend_edges)

print("Проверка трендовых рядов")
print("eps:", EPS_TREND)
print("Исходных рядов:", len(trend_cols))
print("Классов:", len(trend_classes))
print("Сокращение:", len(trend_cols) - len(trend_classes))
print()

for i, members in enumerate(trend_classes, start=1):
    print(f"Класс {i}: {len(members)} рядов")
    print(members)
    print()

,series_1,series_2,distance
0,Lab1_G1_T607,Lab1_TC_T607,0.000000
1,Lab1_G3_T600,Lab1_G3_T1002,0.042885
2,Lab1_PposleNag,Lab1_PdoNag,0.013091
3,Lab1_TposleNag,Lab1_Rc,0.046671


Проверка трендовых рядов
eps: 0.05
Исходных рядов: 13
Классов: 9
Сокращение: 4

Класс 1: 2 рядов
['Lab1_G1_T607', 'Lab1_TC_T607']

Класс 2: 1 рядов
['Lab1_G2_2F2']

Класс 3: 2 рядов
['Lab1_G3_T600', 'Lab1_G3_T1002']

Класс 4: 1 рядов
['Lab1_G4_N1пр']

Класс 5: 1 рядов
['Lab1_Ttg']

Класс 6: 2 рядов
['Lab1_PposleNag', 'Lab1_PdoNag']

Класс 7: 2 рядов
['Lab1_TposleNag', 'Lab1_Rc']

Класс 8: 1 рядов
['Lab1_dev']

Класс 9: 1 рядов
['Lab1_G2_VoГГ']



### Проверка рядов со скачком

Для рядов со скачком сравниваются признаки скачкообразной динамики:

- положение максимального скачка;
- величина и направление скачка;
- абсолютная величина скачка;
- доминирование скачка над обычными изменениями;
- начальный и конечный уровни.

Если при `eps = 0.05` связи похожести не возникают, то такие ряды не объединяются и каждый ряд остаётся отдельным классом.

In [22]:
jump_cols = nonstationary_features[
    nonstationary_features["nonstationary_type"] == "jump"
]["series_name"].tolist()

jump_feature_cols = [
    "jump_pos_rel",
    "jump_size",
    "jump_abs",
    "jump_dominance",
    "start_level",
    "end_level",
]

jump_features = nonstationary_features[
    nonstationary_features["series_name"].isin(jump_cols)
][["series_name"] + jump_feature_cols].copy()

jump_features_norm = min_max_scale(
    jump_features,
    jump_feature_cols
)


def jump_distance(series_1, series_2):
    return mean_abs_distance(
        jump_features_norm,
        jump_feature_cols,
        series_1,
        series_2,
    )


EPS_JUMP = 0.05

jump_classes, jump_edges = build_classes(
    jump_cols,
    jump_distance,
    EPS_JUMP,
)

display(jump_edges)

print("Проверка рядов со скачком")
print("eps:", EPS_JUMP)
print("Исходных рядов:", len(jump_cols))
print("Классов:", len(jump_classes))
print("Сокращение:", len(jump_cols) - len(jump_classes))
print()

for i, members in enumerate(jump_classes, start=1):
    print(f"Класс {i}: {len(members)} рядов")
    print(members)
    print()

""


Проверка рядов со скачком
eps: 0.05
Исходных рядов: 6
Классов: 6
Сокращение: 0

Класс 1: 1 рядов
['Lab1_G3_Pc2']

Класс 2: 1 рядов
['Lab1_TC_Ptgdg']

Класс 3: 1 рядов
['Lab1_dPmg']

Класс 4: 1 рядов
['Lab1_Hpol']

Класс 5: 1 рядов
['Lab1_TdoNag']

Класс 6: 1 рядов
['Lab1_G3_Коксование']



По результатам проверки ряды со скачкообразной динамикой не образуют устойчивых классов похожести при выбранном пороге `eps = 0.05`. Поэтому они не объединяются и остаются отдельными представителями.

Ряды типа `dynamic` на данном этапе дополнительно не факторизуются и также сохраняются как отдельные временные ряды.

In [23]:
# Представители классов, где выполнялась факторизация.

stationary_representatives, stationary_report = choose_representatives(
    stationary_classes,
    stationary_distance,
)
stationary_report["group"] = "stationary"


diff_representatives, diff_report = choose_representatives(
    diff_classes,
    diff_distance,
)
diff_report["group"] = "difference_stationary_or_borderline"


trend_representatives, trend_report = choose_representatives(
    trend_classes,
    trend_distance,
)
trend_report["group"] = "trend"


# Для jump и dynamic ряды оставляются без объединения.
jump_representatives = jump_cols.copy()
dynamic_representatives = dynamic_cols.copy()

jump_report = pd.DataFrame([
    {
        "local_class_id": i,
        "representative": col,
        "class_size": 1,
        "mean_distance_representative": 0.0,
        "members": [col],
        "group": "jump",
    }
    for i, col in enumerate(jump_cols, start=1)
])

dynamic_report = pd.DataFrame([
    {
        "local_class_id": i,
        "representative": col,
        "class_size": 1,
        "mean_distance_representative": 0.0,
        "members": [col],
        "group": "dynamic",
    }
    for i, col in enumerate(dynamic_cols, start=1)
])

In [24]:
factorization_report = pd.concat(
    [
        stationary_report,
        diff_report,
        trend_report,
        jump_report,
        dynamic_report,
    ],
    ignore_index=True,
)

factorization_report.insert(
    0,
    "global_class_id",
    range(1, len(factorization_report) + 1),
)

factorization_report = factorization_report[
    [
        "global_class_id",
        "group",
        "representative",
        "class_size",
        "mean_distance_representative",
        "members",
    ]
]

display(factorization_report)

print("Итоговое количество классов без константных рядов:", len(factorization_report))

,global_class_id,group,representative,class_size,mean_distance_representative,members
0,1,stationary,Lab1_G1_N3,1,0.000000,[Lab1_G1_N3]
1,2,stationary,Lab1_G2_Fн9,12,0.053783,"[Lab1_G2_Fc2, Lab1_G3_V2, Lab1_G3_КВД, Lab1_G3..."
2,3,stationary,Lab1_G2_Fc3,3,0.015828,"[Lab1_G2_Fc3, Lab1_G2_Fв9, Lab1_G2_2F3]"
3,4,stationary,Lab1_G3_Турбина_ГГ,7,0.040979,"[Lab1_G2_3F1, Lab1_G3_Турбина_ГГ, Lab1_G3_ПО_С..."
4,5,difference_stationary_or_borderline,Lab1_G1_N1,1,0.000000,[Lab1_G1_N1]
5,6,difference_stationary_or_borderline,Lab1_G1_N2,2,0.000000,"[Lab1_G1_N2, Lab1_G4_N2]"
6,7,difference_stationary_or_borderline,Lab1_G2_3F2,29,0.027619,"[Lab1_G1_P2, Lab1_Ne, Lab1_PologenieTRK, Lab1_..."
7,8,difference_stationary_or_borderline,Lab1_G1_T4ср,2,0.017443,"[Lab1_G1_T4ср, Lab1_G4_T4]"
8,9,difference_stationary_or_borderline,Lab1_G3_T606,1,0.000000,[Lab1_G3_T606]
9,10,difference_stationary_or_borderline,Lab1_Qtg,1,0.000000,[Lab1_Qtg]


Итоговое количество классов без константных рядов: 31


In [25]:
final_representatives = (
    stationary_representatives
    + diff_representatives
    + trend_representatives
    + jump_representatives
    + dynamic_representatives
)

# Удаление возможных дублей при сохранении порядка.
final_representatives = list(dict.fromkeys(final_representatives))
data_clean = df[["TS"] + final_representatives].copy()

data_clean.to_csv(
    "data_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

data_clean.to_parquet(
    "data_clean.parquet",
    index=False
)

factorization_report.to_csv(
    "factorization_report_without_constants.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Сохранены файлы:")
print("data_clean.csv")
print("data_clean.parquet")
print("factorization_report_without_constants.csv")

Сохранены файлы:
data_clean.csv
data_clean.parquet
factorization_report_without_constants.csv


In [26]:
import ast
import plotly.express as px
import pandas as pd
import numpy as np


def parse_members(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return [value]

    return [value]


factorization_report["members"] = factorization_report["members"].apply(parse_members)

group_summary = (
    factorization_report
    .groupby("group")
    .agg(
        classes_count=("global_class_id", "count"),
        represented_series_count=("class_size", "sum"),
    )
    .reset_index()
)

display(group_summary)


,group,classes_count,represented_series_count
0,difference_stationary_or_borderline,6,36
1,dynamic,6,6
2,jump,6,6
3,stationary,4,23
4,trend,9,13


In [27]:
def normalize_to_01(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    amplitude = values.max() - values.min()

    if amplitude == 0:
        return np.zeros_like(values)

    return (values - values.min()) / amplitude


def plot_class(class_id):
    row = factorization_report[
        factorization_report["global_class_id"] == class_id
    ].iloc[0]

    members = row["members"]
    representative = row["representative"]
    group = row["group"]

    plot_rows = []

    for col in members:
        if col not in df.columns:
            continue

        y = normalize_to_01(df[col])

        for t, value in enumerate(y):
            plot_rows.append({
                "t": t,
                "value": value,
                "series": col,
                "role": "representative" if col == representative else "member",
            })

    plot_df = pd.DataFrame(plot_rows)

    fig = px.line(
        plot_df,
        x="t",
        y="value",
        color="series",
        title=f"Класс {class_id} | группа: {group} | размер: {len(members)}",
        labels={
            "t": "Номер наблюдения",
            "value": "Нормированное значение",
            "series": "Ряд",
        },
    )

    fig.show()

In [209]:
nontrivial_classes = factorization_report[
    factorization_report["class_size"] > 1
]["global_class_id"].tolist()

print("Классы, где было объединено больше одного ряда:")
print(nontrivial_classes)

for class_id in nontrivial_classes:
    plot_class(class_id)

Классы, где было объединено больше одного ряда:
[2, 3, 4, 6, 7, 8, 11, 13, 16, 17]


In [ ]:
import ast
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output


def parse_members(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            return [value]

    return [value]


def normalize_to_01(values):
    values = pd.Series(values).dropna().to_numpy(dtype=float)

    amplitude = values.max() - values.min()

    if amplitude == 0:
        return np.zeros_like(values)

    return (values - values.min()) / amplitude


factorization_report["members"] = factorization_report["members"].apply(parse_members)

class_options = []

for _, row in factorization_report.iterrows():
    label = (
        f"Класс {row['global_class_id']} | "
        f"{row['group']} | "
        f"{row['representative']} | "
        f"размер: {row['class_size']}"
    )
    class_options.append((label, row["global_class_id"]))


class_dropdown = widgets.Dropdown(
    options=class_options,
    description="Класс:",
    layout=widgets.Layout(width="90%")
)

plot_button = widgets.Button(
    description="Построить график",
    button_style="primary"
)

output = widgets.Output()


def plot_selected_class(class_id, step=5):
    row = factorization_report[
        factorization_report["global_class_id"] == class_id
    ].iloc[0]

    members = row["members"]
    representative = row["representative"]
    group = row["group"]

    plot_rows = []

    for col in members:
        if col not in df.columns:
            continue

        y = normalize_to_01(df[col])

        for t in range(0, len(y), step):
            plot_rows.append({
                "t": t,
                "value": y[t],
                "series": col,
                "role": "representative" if col == representative else "member",
            })

    plot_df = pd.DataFrame(plot_rows)

    fig = px.line(
        plot_df,
        x="t",
        y="value",
        color="series",
        line_dash="role",
        title=(
            f"Класс {class_id} | {group} | "
            f"представитель: {representative} | размер: {len(members)}"
        ),
        labels={
            "t": "Номер наблюдения",
            "value": "Нормированное значение",
            "series": "Ряд",
            "role": "Роль",
        },
    )

    fig.update_layout(height=500)
    return fig


def on_button_click(_):
    with output:
        clear_output(wait=True)

        class_id = class_dropdown.value
        fig = plot_selected_class(class_id, step=5)

        fig.show()


plot_button.on_click(on_button_click)

display(class_dropdown, plot_button, output)

Dropdown(description='Класс:', layout=Layout(width='90%'), options=(('Класс 1 | stationary | Lab1_G1_N3 | разм…

Button(button_style='primary', description='Построить график', style=ButtonStyle())

Output()

In [8]:
print(f"Найдем колонки где данные принимают только 1 значение.")
# Количество уникальных значений в каждой колонке
nunique = df.nunique(dropna=False)

# Колонки, где только одно уникальное значение
constant_cols = nunique[nunique == 1].index.tolist()

print(f"Константных колонок: {len(constant_cols)}")
constant_cols

df_for_visualization = df.drop(columns=constant_cols)

print(f"Было колонок: {df.shape[1]}")
print(f"Стало колонок: {df_for_visualization.shape[1]}")
print(f"Удалено константных колонок: {len(constant_cols)}")

visualization_cols = df_for_visualization.columns

batch_size = 10

for i in range(0, len(visualization_cols), batch_size):
    cols_batch = visualization_cols[i:i + batch_size]

    fig = px.line(
        df_for_visualization,
        x="TS",
        y=cols_batch,
        title=f"Временные ряды {i + 1}-{i + len(cols_batch)}",
    )

    fig.update_layout(
        height=600,
        xaxis_title="Время",
        yaxis_title="Значение",
        legend_title="Показатель",
    )

    fig.show()
    
constant_values = (
    df[constant_cols]
    .iloc[0]
    .reset_index()
)

constant_values.columns = ["column", "constant_value"]

constant_values_sorted = constant_values.sort_values(
    "constant_value",
    ascending=True
)

fig = px.bar(
    constant_values_sorted,
    x="column",
    y="constant_value",
    title="Константные признаки и их значения",
)

fig.update_layout(
    height=500,
    xaxis_title="Признак",
    yaxis_title="Постоянное значение",
    xaxis_tickangle=-45,
)

fig.show()

Найдем колонки где данные принимают только 1 значение.
Константных колонок: 33
Было колонок: 118
Стало колонок: 85
Удалено константных колонок: 33


## Сохраним dataset после предобработки

In [10]:
df.to_parquet("dataset_clean.parquet", index=False)
df.to_csv("dataset_clean.csv", index=False, encoding="utf-8-sig")